# Model splitting for parallel and serial MODFLOW 6

**Model splitting** (also called *partitioning*) takes one MODFLOW 6 simulation and divides its grid into several smaller submodels that, run together, reproduce the original model's results. FloPy does this for you with the `Mf6Splitter()` class: it maps the original model's cell-to-cell connectivity, assigns every cell to a submodel using an array of model numbers you provide, and then builds the new submodels — automatically adding the **exchanges** (and any **movers**) that pass flow across the cuts so the pieces stay hydraulically connected.

Why split a model? Two main reasons:

- **Parallelization** — each submodel can be solved by a separate process (for example with Message Passing Interface, **MPI**), so a large simulation runs faster across multiple cores.
- **Memory** — no single process has to hold the entire grid in memory at once.

`Mf6Splitter()` works on groundwater flow (**GWF**) models, on combined flow-and-transport (**GWF/GWT**) models, and on Structured, Vertex, and Unstructured grids.

## What this notebook covers

You will:

- load and run an existing simulation, split it into **five blocks of roughly equal size** (one holding the river), and confirm the split results match the original;
- let `Mf6Splitter()` build a *load-balanced* partition automatically for a chosen number of submodels; and
- save and reload the **node mapping** so a split model's output can be stitched back onto the original grid for plotting and comparison.

The first example uses the classic Freyberg (1988) groundwater flow model.

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm

## Example 1: splitting a simple structured grid model

This example shows the basics of using the `Mf6Splitter()` class and applies the method to the Freyberg (1988) model.

In [ ]:
simulation_ws = Path("../data/mf6-freyberg")

Load the existing Freyberg simulation from disk with `flopy.mf6.MFSimulation.load()`, pointing `sim_ws` at the folder that holds its input files. This gives you a complete simulation object to run and then split.

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=simulation_ws)

Point the simulation at a fresh working directory with `sim.set_sim_path()`, write the input files with `sim.write_simulation()`, and run MODFLOW 6 with `sim.run_simulation()`. Running the original model first gives you a baseline to compare the split results against.

In [ ]:
# point the loaded simulation at a fresh workspace, then write and run it
workspace = Path("models/freyberg")
sim.set_sim_path(workspace)
sim.write_simulation()
success, buff = sim.run_simulation()
assert success, "MODFLOW 6 did not terminate normally"

#### Plot the original model

Get the flow model from the simulation and load its final head array, then map it with `flopy.plot.PlotMapView()`. Draw the heads with `.plot_array()` and the boundary-condition cells — wells (`WELL`), river (`RIV`), and constant head (`CHD`) — with `.plot_bc()`. The `1e30` values MODFLOW writes for dry or inactive cells are replaced with `NaN` so they do not distort the color scale.

In [ ]:
# get the flow model; the plotting cell below loads its final heads
gwf = sim.get_model()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 7))
pmv = flopy.plot.PlotMapView(gwf, ax=ax)
heads = gwf.output.head().get_alldata()[-1]
heads = np.where(heads == 1e30, np.nan, heads)
vmin = np.nanmin(heads)
vmax = np.nanmax(heads)
pc = pmv.plot_array(heads, vmin=vmin, vmax=vmax)
pmv.plot_bc("WEL")
pmv.plot_bc("RIV", color="c")
pmv.plot_bc("CHD")
pmv.plot_grid()
pmv.plot_ibound()
plt.colorbar(pc);

**What to look for.** This is the un-split, original model — your baseline. Heads grade across the domain, with the river and constant-head boundaries setting the levels and the well cells marking local stresses. Keep this map in mind: after splitting and recombining, the reassembled results should reproduce it.

### Define the split with an array

Splitting is driven by an integer array the size of one layer (`StructuredGrid`, `VertexGrid`) or the node count (`UnstructuredGrid`). Each value is the submodel a cell joins, and every submodel's cells must be **contiguous**.

Here you split the grid into **five blocks with roughly equal active-cell counts** (~141 each). Balance on **active** cells only — inactive cells (`idomain < 1`) still get a block number, but MODFLOW ignores them. Three stacked bands cover the land left of the river (their row cuts computed from the active-cell counts), a central strip holds **every river cell** (column 14), and the last block is the land to the right. The river's own block keeps the whole river in one submodel.

In [ ]:
modelgrid = gwf.modelgrid

In [ ]:
# balance the split on ACTIVE cells only (idomain < 1 cells are inactive)
idomain = modelgrid.idomain[0]
nrow, ncol = idomain.shape

# columns 0-11 are land left of the river, 12-15 the river strip, 16-19 land right
left = slice(0, 12)

# split the left land into three stacked bands with ~equal active-cell counts
active_per_row = (idomain[:, left] == 1).sum(axis=1)
cum = np.cumsum(active_per_row)
r1, r2 = np.searchsorted(cum, [cum[-1] / 3, 2 * cum[-1] / 3])

array = np.zeros((nrow, ncol), dtype=int)
array[:r1, left] = 1  # left land, top band
array[r1:r2, left] = 2  # left land, middle band
array[r2:, left] = 3  # left land, bottom band
array[:, 12:16] = 4  # central strip - holds every river cell (column 14)
array[:, 16:20] = 5  # land right of the river

for k in range(1, 6):
    print(f"block {k}: {int(((array == k) & (idomain == 1)).sum())} active cells")

Preview the partition with `flopy.plot.PlotMapView().plot_array()`, masking inactive cells so only active cells take a submodel color. Use a discrete colormap (one color per submodel, ticks centered on each integer), and overlay the inactive cells (`.plot_inactive()`) and the boundary conditions — wells (`WEL`), river (`RIV`), and constant head (`CHD`).

In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))
pmv = flopy.plot.PlotMapView(gwf, ax=ax)

# one discrete color per submodel, with ticks centered on each integer
labels = np.unique(array)
cmap = plt.get_cmap("Dark2", labels.size)
norm = BoundaryNorm(np.arange(labels.min() - 0.5, labels.max() + 1.5), cmap.N)

# color only the active cells; draw the inactive cells separately
pc = pmv.plot_array(np.ma.masked_where(idomain < 1, array), cmap=cmap, norm=norm)
pmv.plot_inactive()
pmv.plot_bc("WEL")
pmv.plot_bc("RIV", color="c")
pmv.plot_bc("CHD")
pmv.plot_grid(lw=0.3)
plt.colorbar(pc, ticks=labels, label="submodel")
plt.show()

**What to look for.** Five colors, five submodels. The lines between colors are where `Mf6Splitter()` cuts the grid and inserts exchanges. Each block is one contiguous piece with a similar active-cell count, and the river (cyan) lies entirely inside the central block — so no cut crosses it.

### Splitting the model using `Mf6Splitter()`

The `Mf6Splitter()` class accepts one required parameter and one optional parameter. These parameters are:
   - `sim`: A flopy.mf6.MFSimulation object
   - `modelname`: optional, the name of the model being split. If omitted Mf6Splitter grabs the first groundwater flow model listed in the simulation

In [ ]:
# create the splitter for the loaded simulation
from flopy.mf6.utils import Mf6Splitter

mfs = Mf6Splitter(sim)

The model splitting is then performed by calling the `split_model()` function. `split_model()` accepts an array that is either the same size as the number of cells per layer (`StructuredGrid` and `VertexGrid`) model or the number of nodes in the model (`UnstructuredGrid`).

_Note:This method also accepts an optional `sim_ws` parameter. This parameter is useful when splitting a model that has external files and you'd like to preserve the external structure. The `sim_ws` parameter is the path to where you'd like to save the split simulation._

This function returns a new `MFSimulation` object that contains the split models and exchanges between them

In [ ]:
# split on the model-number array; returns a new simulation with both submodels
new_sim = mfs.split_model(array)
new_sim.set_sim_path(Path("models/freyberg_split"))

In [ ]:
# now to write and run the simulation
new_sim.write_simulation()
success, buff = new_sim.run_simulation()
assert success, "MODFLOW 6 did not terminate normally"

#### Compare the split models with the original

Pull each submodel from `new_sim` with `.get_model()`, load its final heads, and draw all five on one map, reusing the original `vmin`/`vmax`. Because the submodels share the original coordinate system, plotted together they redraw the original head field.

In [ ]:
# collect the five submodels
submodels = {name: new_sim.get_model(name) for name in new_sim.model_names}

In [ ]:
# final heads for each submodel, with dry/inactive (1e30) blanked out
heads = {}
for name, ml in submodels.items():
    h = ml.output.head().get_alldata()[-1]
    heads[name] = np.where(h == 1e30, np.nan, h)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 7))
extent = gwf.modelgrid.extent  # pin every panel to the full domain
for name, ml in submodels.items():
    pmv = flopy.plot.PlotMapView(ml, ax=ax, extent=extent)
    pc = pmv.plot_array(heads[name], vmin=vmin, vmax=vmax)
    pmv.plot_grid(lw=0.3)
# overlay the river from the original model to show it sits in one block
flopy.plot.PlotMapView(gwf, ax=ax, extent=extent).plot_bc("RIV", color="c")
ax.set_title("Five submodels, solved together")
plt.colorbar(pc, ax=ax, shrink=0.6, label="Hydraulic heads");

**What to look for.** The five submodels were solved together as one simulation. Plotted on the shared grid they reproduce the original single-model map — heads run continuously across every cut, with no seam, because the exchanges pass flow between the pieces. Splitting changes *how* MODFLOW 6 solves the problem, not the answer.

## Example 2: build a load-balanced mask automatically

In Example 1 you drew the mask by hand. `Mf6Splitter.optimize_splitting_mask()` builds one for you: it weights an adjacency graph by the active and inactive nodes in every layer, hands it to `pymetis`, and returns a balanced membership array. You just give it the number of parts.

- `nparts` — **required**, int; number of submodels (e.g. `nparts=5`)
- `active_only` — optional, bool; balance on active cells only
- `options` — optional, `pymetis.Options` passed through to pymetis (e.g. `pymetis.Options(seed=42)`)
- `verbose` — optional, bool; print progress

In [ ]:
# build a load-balanced partition automatically (uses pymetis under the hood)
mfs = Mf6Splitter(sim)
array = mfs.optimize_splitting_mask(nparts=5)

Map the optimized model-number array the same way — active cells only, one discrete color per submodel, with the inactive cells and boundary conditions overlaid — to preview the automatically generated partition.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))
pmv = flopy.plot.PlotMapView(gwf, ax=ax)

# same discrete colormap and overlays as the manual split
labels = np.unique(array)
cmap = plt.get_cmap("Dark2", labels.size)
norm = BoundaryNorm(np.arange(labels.min() - 0.5, labels.max() + 1.5), cmap.N)

pc = pmv.plot_array(np.ma.masked_where(idomain < 1, array), cmap=cmap, norm=norm)
pmv.plot_inactive()
pmv.plot_bc("WEL")
pmv.plot_bc("RIV", color="c")
pmv.plot_bc("CHD")
pmv.plot_grid(lw=0.3)
plt.colorbar(pc, ticks=labels, label="submodel")
plt.show()

**What to look for.** `optimize_splitting_mask()` chose these partitions for you rather than you drawing them by hand. Each color is one submodel, and the pieces are sized to hold roughly equal numbers of active cells so the parallel workload stays balanced across processes. Because it balances on cell counts alone, it does not keep the river (cyan) in a single block the way the hand-drawn split did.

Split the model on the optimized mask with `mfs.split_model()`, point the returned simulation at a workspace with `set_sim_path()`, then write and run it.

In [ ]:
sim_ws = Path("models/load_balanced_split")
new_sim = mfs.split_model(array)
new_sim.set_sim_path(sim_ws)
new_sim.write_simulation()
success, buff = new_sim.run_simulation()
assert success, "MODFLOW 6 did not terminate normally"

## Saving the node mapping to file

The **node mapping** records which original-grid cell each submodel cell came from. You need it to reassemble split output back onto the original grid, so it is worth saving alongside the split model. Save it with `Mf6Splitter.save_node_mapping()`, which writes the mapping to a Hierarchical Data Format version 5 (**HDF5**) binary file.

In [ ]:
# save the node mapping alongside the split model (HDF5)
mfs.save_node_mapping(sim_ws / "node_map.hdf5")

## Loading a saved node mapping from file

Reload a saved mapping — for example in a fresh session, after the split model has already been run — with `Mf6Splitter.load_node_mapping()`, passing it the HDF5 file written above. With the mapping loaded, the splitter can reconstruct arrays on the original grid without re-running the split.

In [ ]:
new_sim2 = flopy.mf6.MFSimulation.load(sim_ws=sim_ws)

# load_node_mapping is a staticmethod that returns a ready-to-use Mf6Splitter
mfsplit = Mf6Splitter.load_node_mapping(sim_ws / "node_map.hdf5")

#### Reassemble and plot the split results

Loop over the submodels, collecting each one's final head array into a dictionary keyed by submodel index, then stitch them back onto the original grid with `mfsplit.reconstruct_array()`. This is where the node mapping earns its keep: it tells `reconstruct_array()` which original cell each submodel cell belongs to. Plot the reconstructed head array on the original model grid.

In [ ]:
head_dict = {}
for ix, mname in enumerate(new_sim2.model_names):
    ml = new_sim2.get_model(mname)
    head_dict[ix] = ml.output.head().get_alldata()[-1]

ra_heads = mfsplit.reconstruct_array(head_dict)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 8))

pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, ax=ax)
pc = pmv.plot_array(ra_heads)
ib = pmv.plot_inactive()
plt.colorbar(pc, shrink=0.8);

**What to look for.** The submodel heads have been stitched back onto the single original grid, so this map should look like the baseline map from the start of the notebook — with no visible trace of where the submodel boundaries were. That agreement confirms the split-and-recombine round trip preserved the solution.

More information about the model splitter can be found [here](https://flopy.readthedocs.io/en/latest/Notebooks/mf6_parallel_model_splitting_example.html)

## Recap

In this notebook you:

- loaded and ran the original Freyberg (1988) flow model as a baseline;
- used `Mf6Splitter().split_model()` to split it into five contiguous blocks of roughly equal size — one block holding the entire river — letting FloPy add the exchanges that keep the pieces connected, and confirmed the split results match the original;
- had `optimize_splitting_mask()` build a *load-balanced* partition automatically for a chosen number of submodels; and
- saved the **node mapping** and used `reconstruct_array()` to stitch a split model's heads back onto the original grid for comparison.

The key idea: splitting changes *how* MODFLOW 6 solves a simulation — enabling parallel runs and lower per-process memory — without changing the answer.